# Feature Generation:

This step turns sense/antisense (24 nt) sequences and efficacy into machine-learning–ready features. It creates one-hot encodings and biologically meaningful k-mer and composition features (mono/di/tri-nucleotide), binary chemophysical patterns, and GC metrics, then saves train/validation matrices.

Input: TSV with columns sense_24len, antisense_24len, Efficiacy (24-nt, lowercase a/t/g/c/x with z padding already applied if needed).

Outputs:

X_train.npy, y_train.npy, X_val.npy, y_val.npy (or .tsv/.parquet if preferred)

Optional per-feature dictionary: feature_names.json

Feature blocks (per strand, then concatenated sense||antisense):

One-hot (positional): 24×5 for a,t,g,c,x/z per strand → flattened.

Mononucleotide counts & fractions: counts and % of A,T,G,C,X (±Z); overall GC content.

Dinucleotide (k=2) counts/fractions: all 25 bigrams over ATGCX (or 16 over ATGC if you exclude X/Z).

Trinucleotide (k=3) counts/fractions: 125 (or 64 for ATGC only), optionally top-N most frequent across corpus to reduce dimensionality.

Binary chemophysical patterns (per position + global %):

Purine/Pyrimidine: R={A,G} vs Y={C,T}

Strong/Weak H-bond: S={G,C} vs W={A,T}

Amino/Keto: M={A,C} vs K={G,T}

Unknown/pad flags: X (unknown) and Z (pad) masks

Motif flags (optional): presence counts of selected inhibitory/efficacious motifs (e.g., UU, GGG, seed-like patterns).

Preprocessing assumptions:

Sequences are 24 nt, lowercase, with only a/t/g/c/x/z. Any other char will be mapped to x during featurization.

If you use separate train/independent validation files, we fit any vocab/selection on train only, then apply to validation.

Configuration knobs:

Alphabet (ATGC vs ATGCX), whether to keep Z (pad) as its own channel.

K-mer orders to include (k=1/2/3), normalize as counts or freq.

Dimensionality control: keep full tri-mer space or top-N tri-mers by corpus frequency.

Usage flow:

Load train/val TSVs.

Sanity map to constrained alphabet and assert length = 24.

Build per-strand features (one-hot, k-mers, binary patterns, GC).

Concatenate sense || antisense features; align y from Efficiacy.

Save X_*, y_* and feature_names.

# Step 1 — Imports & configuration

In [1]:
import itertools
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# Fixed assumptions from your preprocessing:
# - sequences are 24 nt, lowercase, alphabet subset of {a,t,g,c,x,z}
# - 'z' is right-padding; keep it as its own channel (set True) or merge with 'x' (False)
KEEP_Z_CHANNEL = True

ALPH_CORE = "atgc"        # canonical nucleotides
ALPH_UNKNOWN = "x"        # unknown char from cleaning
ALPH_PAD = "z"            # pad char
ALPH_ONEHOT = ALPH_CORE + ALPH_UNKNOWN + (ALPH_PAD if KEEP_Z_CHANNEL else "")
ALPH_KMER = ALPH_CORE + ALPH_UNKNOWN     # we will EXCLUDE 'z' from k-mer counting
SEQ_LEN = 24


/Users/darsa/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Step 2 — Load your data and check

1) T/t are converted to U.

2) The final alphabet is guaranteed to be {a,c,g,u,x,z}.

3) Every sequence is exactly 24 nt long.

In [2]:
# Change file path as per need
in_path = Path("../data/D_sirnas_90percent_input.txt")
df = pd.read_csv(in_path, sep="\t", dtype=str)


ALLOWED = set("acguxz")
TARGET_LEN = 24

def sanitize_rna(seq: str, target_len: int = TARGET_LEN) -> str:
    if not isinstance(seq, str):
        seq = ""
    s = seq.lower()
    s = s.replace("t", "u")                  # convert DNA T to RNA U
    s = re.sub(r"[^acguxz]", "x", s)         # map anything else -> x
    if len(s) < target_len:
        s = s + "z" * (target_len - len(s))  # right-pad with z
    return s[:target_len]                     # truncate if longer

# Apply to your dataframe
df["sense"] = df["sense_24len"].apply(sanitize_rna)
df["antisense"] = df["antisense_24len"].apply(sanitize_rna)

df

,sense_24len,antisense_24len,Efficiacy,sense,antisense
0,gacgxaaacggccacaagxtcxzz,acxxgxggccgxxxacgxcgczzz,67.1,gacgxaaacggccacaagxucxzz,acxxgxggccgxxxacgxcgczzz
1,gaccacgaagagxxaacccttzzz,gggxxaacxcxxcgxggxcttzzz,85,gaccacgaagagxxaacccuuzzz,gggxxaacxcxxcgxggxcuuzzz
2,gacgtaaacggccacaagxtcxzz,acxxgxggccgxxxacgxcgczzz,36.7,gacguaaacggccacaagxucxzz,acxxgxggccgxxxacgxcgczzz
3,gcagcacgacxxcxxcaagttzzz,cxxgaagaagxcgxgcxgcttzzz,0,gcagcacgacxxcxxcaaguuzzz,cxxgaagaagxcgxgcxgcuuzzz
4,cxxacgcxgagxacxxcgattzzz,xcgaagxacxcagcgxaagttzzz,51,cxxacgcxgagxacxxcgauuzzz,xcgaagxacxcagcgxaaguuzzz
...,...,...,...,...,...
3986,ccagxaaggcxxcxcxxaaxxzzz,xxaagagaagccxxacxggxxzzz,98.4,ccagxaaggcxxcxcxxaaxxzzz,xxaagagaagccxxacxggxxzzz
3987,gacgtaaacggccacaagttctzz,acxxgxggccgxxxacgxcgcxzz,10.2,gacguaaacggccacaaguucuzz,acxxgxggccgxxxacgxcgcxzz
3988,gacgxaaacggccacaagxxtzzz,acxxgxggccgxxxacgxcgtxzz,80.8,gacgxaaacggccacaagxxuzzz,acxxgxggccgxxxacgxcguxzz
3989,gacgxaaacggccacaagxtcxzz,acxxgxggccgxxxacgxcgczzz,88.2,gacgxaaacggccacaagxucxzz,acxxgxggccgxxxacgxcgczzz


# Next we will create -- One hot encoding

One-Hot Encoding of siRNA Sequences
What is One-Hot Encoding?

One-hot encoding is a way to represent categorical variables as binary vectors.
For sequences like siRNA, each nucleotide at each position is categorical (a, c, g, u, x, z).

Each base is mapped to a binary vector where only one entry is 1 and all others are 0.

For a fixed sequence length (24 in our case), you create one binary vector per position, then concatenate them to form a full sequence representation.

Stack them position by position:

Position	Base	One-Hot Vector

1	a	1 0 0 0 0 0

2	c	0 1 0 0 0 0

3	g	0 0 1 0 0 0

4	u	0 0 0 1 0 0

5	z	0 0 0 0 0 1

6	x	0 0 0 0 1 0

Each siRNA has two strands (sense and antisense).
We generate one-hot features for both, then concatenate:

Sense strand → 144 features

Antisense strand → 144 features

Final feature vector per siRNA = 288 features

In [4]:
sense_seq     = "acguacguacguacguacguacgu"
antisense_seq = "ugcaugcaugcaugcaugcaugca"


ALPHABET = "acguxz"
IDX = {ch:i for i,ch in enumerate(ALPHABET)}

def one_hot_seq(seq: str) -> np.ndarray:
    arr = np.zeros((len(seq), len(ALPHABET)), dtype=int)
    for i, ch in enumerate(seq):
        if ch in IDX:
            arr[i, IDX[ch]] = 1
    return arr

# Example: one siRNA (first row)
sense_oh = one_hot_seq(sense_seq)
anti_oh  = one_hot_seq(antisense_seq)

print("Sense one-hot shape:", sense_oh.shape)
print("Antisense one-hot shape:", anti_oh.shape)
print("Flattened feature size per strand:", sense_oh.size)

print("\nSense one-hot (first 5 positions):")
print(sense_oh[:5])

print("\nAntisense one-hot (first 5 positions):")
print(anti_oh[:5])



Sense one-hot shape: (24, 6)
Antisense one-hot shape: (24, 6)
Flattened feature size per strand: 144

Sense one-hot (first 5 positions):
[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [1 0 0 0 0 0]]

Antisense one-hot (first 5 positions):
[[0 0 0 1 0 0]
 [0 0 1 0 0 0]
 [0 1 0 0 0 0]
 [1 0 0 0 0 0]
 [0 0 0 1 0 0]]


# k-mer Counting for siRNA Sequences

K-mer counting is a common way to represent nucleotide sequences as numerical feature vectors. A k-mer is simply a contiguous substring of length k (e.g., AC, GU, ACG). By sliding a window across each sequence, we count how many times each possible k-mer occurs. These counts become features that summarize motif frequencies in siRNA sequences.

Alphabet Used

We work with the RNA alphabet a, c, g, u, plus x (unknown/uncertain base).

Padding (z) is excluded from k-mer counting windows.

Thus, the effective alphabet size = 5.

Feature Spaces

Monomers (k=1): 5 possible k-mers (a,c,g,u,x) → 5 features.

Dimers (k=2): 25 possible k-mers (5²) → 25 features.

Trimers (k=3): 125 possible k-mers (5³) → 125 features.

In [5]:
def kmer_vocab(k):
    return ["".join(p) for p in itertools.product("acgux", repeat=k)]

def kmer_counts(seq, k):
    vocab = kmer_vocab(k)
    idx = {kmer:i for i,kmer in enumerate(vocab)}
    counts = np.zeros(len(vocab), dtype=int)
    for i in range(len(seq)-k+1):
        w = seq[i:i+k]
        if "z" not in w:        # ignore padded segments
            counts[idx[w]] += 1
    return counts, vocab

sense_seq     = "acguacguacguacguacguacgu"
antisense_seq = "ugcaugcaugcaugcaugcaugca"

# Sense strand counts
mono_counts, mono_vocab = kmer_counts(sense_seq, 1)
di_counts, di_vocab     = kmer_counts(sense_seq, 2)
tri_counts, tri_vocab   = kmer_counts(sense_seq, 3)

print("Sense — mono k-mers (nonzero):", {k:v for k,v in zip(mono_vocab, mono_counts) if v>0})
print("Sense — di k-mers (nonzero):",   {k:v for k,v in zip(di_vocab, di_counts) if v>0})
print("Sense — tri k-mers (nonzero):",  {k:v for k,v in zip(tri_vocab, tri_counts) if v>0})
print("Feature sizes: mono:", len(mono_counts),
      "di:", len(di_counts),
      "tri:", len(tri_counts))

# Antisense strand counts
mono_counts_a, mono_vocab_a = kmer_counts(antisense_seq, 1)
di_counts_a, di_vocab_a     = kmer_counts(antisense_seq, 2)
tri_counts_a, tri_vocab_a   = kmer_counts(antisense_seq, 3)

print("\nAntisense — mono k-mers (nonzero):", {k:v for k,v in zip(mono_vocab_a, mono_counts_a) if v>0})
print("Antisense — di k-mers (nonzero):",   {k:v for k,v in zip(di_vocab_a, di_counts_a) if v>0})
print("Antisense — tri k-mers (nonzero):",  {k:v for k,v in zip(tri_vocab_a, tri_counts_a) if v>0})
print("Feature sizes: mono:", len(mono_counts_a),
      "di:", len(di_counts_a),
      "tri:", len(tri_counts_a))


Sense — mono k-mers (nonzero): {'a': 6, 'c': 6, 'g': 6, 'u': 6}
Sense — di k-mers (nonzero): {'ac': 6, 'cg': 6, 'gu': 6, 'ua': 5}
Sense — tri k-mers (nonzero): {'acg': 6, 'cgu': 6, 'gua': 5, 'uac': 5}
Feature sizes: mono: 5 di: 25 tri: 125

Antisense — mono k-mers (nonzero): {'a': 6, 'c': 6, 'g': 6, 'u': 6}
Antisense — di k-mers (nonzero): {'au': 5, 'ca': 6, 'gc': 6, 'ug': 6}
Antisense — tri k-mers (nonzero): {'aug': 5, 'cau': 5, 'gca': 6, 'ugc': 6}
Feature sizes: mono: 5 di: 25 tri: 125


#  What are Binary Fractions?

Binary fractions are global composition features based on chemical properties of nucleotides.
They don’t encode positions, they encode proportions.

We group nucleotides into biophysical categories:

R / Y: Purines (A,G) vs. Pyrimidines (C,U)

S / W: Strong H-bonds (G,C) vs. Weak H-bonds (A,U)

M / K: Amino group (A,C) vs. Keto group (G,U)

For a sequence, we compute the fraction of bases in each category:

So you always get 6 numbers per strand.

In [6]:
def binary_patterns(seq: str) -> dict:
    """
    Compute fractions of nucleotides in R/Y, S/W, M/K classes.
    Ignores 'z' padding.
    Returns dict with 6 features.
    """
    s = seq.replace("z", "")
    L = len(s) or 1  # avoid division by zero
    
    R = sum(ch in "ag" for ch in s) / L   # Purines (A,G)
    Y = sum(ch in "cu" for ch in s) / L   # Pyrimidines (C,U)
    S = sum(ch in "gc" for ch in s) / L   # Strong (3 H-bonds)
    W = sum(ch in "au" for ch in s) / L   # Weak (2 H-bonds)
    M = sum(ch in "ac" for ch in s) / L   # Amino
    K = sum(ch in "gu" for ch in s) / L   # Keto
    
    return {"frac_R":R, "frac_Y":Y, "frac_S":S,
            "frac_W":W, "frac_M":M, "frac_K":K}

# Compute fractions
sense_bin = binary_patterns(sense_seq)
anti_bin  = binary_patterns(antisense_seq)

print("Sense binary fractions:", sense_bin)
print("Antisense binary fractions:", anti_bin)
print("Feature vector size per strand:", len(sense_bin))
print("Total for sense+antisense:", len(sense_bin)*2)


Sense binary fractions: {'frac_R': 0.5, 'frac_Y': 0.5, 'frac_S': 0.5, 'frac_W': 0.5, 'frac_M': 0.5, 'frac_K': 0.5}
Antisense binary fractions: {'frac_R': 0.5, 'frac_Y': 0.5, 'frac_S': 0.5, 'frac_W': 0.5, 'frac_M': 0.5, 'frac_K': 0.5}
Feature vector size per strand: 6
Total for sense+antisense: 12


# Lets start applying these features on all data and create vectors

In [11]:
# === Build feature matrices from existing df columns ===
sense_col = "sense"
anti_col  = "antisense"
eff_col   = "Efficiacy"

# 1) One-hot (positional): (N, 24*6*2) = (N, 288)
X_sense_oh = np.stack([one_hot_seq(s).ravel() for s in df[sense_col].values], dtype=np.float32)  # (N, 144)
X_anti_oh  = np.stack([one_hot_seq(s).ravel() for s in df[anti_col].values],  dtype=np.float32)  # (N, 144)
X_onehot   = np.concatenate([X_sense_oh, X_anti_oh], axis=1)                                     # (N, 288)

# 2) k-mer counts (mono+di+tri per strand = 155; both strands = 310)
X_sense_km = np.stack([np.concatenate([
                        kmer_counts(s, 1)[0],
                        kmer_counts(s, 2)[0],
                        kmer_counts(s, 3)[0]
                    ]) for s in df[sense_col].values], dtype=np.float32)                         # (N, 155)
X_anti_km  = np.stack([np.concatenate([
                        kmer_counts(s, 1)[0],
                        kmer_counts(s, 2)[0],
                        kmer_counts(s, 3)[0]
                    ]) for s in df[anti_col].values], dtype=np.float32)                          # (N, 155)
X_kmer     = np.concatenate([X_sense_km, X_anti_km], axis=1)                                     # (N, 310)

# 3) Binary chemophysical fractions (R,Y,S,W,M,K per strand = 6; both = 12)
#    (binary_patterns returns a dict → convert inline to a fixed-order vector)
X_sense_bin = np.vstack([
    [d["frac_R"], d["frac_Y"], d["frac_S"], d["frac_W"], d["frac_M"], d["frac_K"]]
    for d in (binary_patterns(s) for s in df[sense_col].values)
]).astype(np.float32)                                                                             # (N, 6)

X_anti_bin = np.vstack([
    [d["frac_R"], d["frac_Y"], d["frac_S"], d["frac_W"], d["frac_M"], d["frac_K"]]
    for d in (binary_patterns(s) for s in df[anti_col].values)
]).astype(np.float32)                                                                             # (N, 6)

X_binary = np.concatenate([X_sense_bin, X_anti_bin], axis=1)                                     # (N, 12)

print("Shapes — onehot:", X_onehot.shape, "kmer:", X_kmer.shape, "binary:", X_binary.shape)

# # 4) Combine for ML
# X_full = np.concatenate([X_onehot, X_kmer, X_binary], axis=1)                                    # (N, 610)
# y = df[eff_col].astype(float).to_numpy()                                                         # (N,)

# Quick checks
print("X_full:", X_full.shape, " | y:", y.shape)
print("Per-row lengths -> onehot:", X_onehot.shape[1],
      "kmer:", X_kmer.shape[1], "binary:", X_binary.shape[1], "full:", X_full.shape[1])

# Optional: save for modeling
# np.save("X_onehot.npy", X_onehot)
# np.save("X_kmer.npy",   X_kmer)
# np.save("X_binary.npy", X_binary)
# np.save("X_full.npy",   X_full)
# np.save("y.npy",        y)


Shapes — onehot: (3991, 288) kmer: (3991, 310) binary: (3991, 12)
X_full: (3991, 610)  | y: (3991,)
Per-row lengths -> onehot: 288 kmer: 310 binary: 12 full: 610


In [12]:
# One-hot + k-mer
X_oh_km = np.concatenate([X_onehot, X_kmer], axis=1)
print("X_oh_km:", X_oh_km.shape)

# One-hot + binary
X_oh_bin = np.concatenate([X_onehot, X_binary], axis=1)
print("X_oh_bin:", X_oh_bin.shape)

# k-mer + binary
X_km_bin = np.concatenate([X_kmer, X_binary], axis=1)
print("X_km_bin:", X_km_bin.shape)

# All three (already done before)
X_full = np.concatenate([X_onehot, X_kmer, X_binary], axis=1)
print("X_full:", X_full.shape)


X_oh_km: (3991, 598)
X_oh_bin: (3991, 300)
X_km_bin: (3991, 322)
X_full: (3991, 610)


In [13]:
# Pick the first row
row_idx = 0

vec_onehot = X_onehot[row_idx]
vec_kmer   = X_kmer[row_idx]
vec_binary = X_binary[row_idx]
vec_full   = X_full[row_idx]

print("Row index:", row_idx)
print("One-hot length:", vec_onehot.size)
print("k-mer length:", vec_kmer.size)
print("Binary length:", vec_binary.size)
print("Full vector length:", vec_full.size)

# Print the full vector (all features combined)
print("\nFull feature vector (first row):")
print(vec_full)

# Optional: if it's too long, show only the beginning
print("\nFull vector (first 50 features):")
print(vec_full[:50])


Row index: 0
One-hot length: 288
k-mer length: 310
Binary length: 12
Full vector length: 610

Full feature vector (first row):
[0.         0.         1.         0.         0.         0.
 1.         0.         0.         0.         0.         0.
 0.         1.         0.         0.         0.         0.
 0.         0.         1.         0.         0.         0.
 0.         0.         0.         0.         1.         0.
 1.         0.         0.         0.         0.         0.
 1.         0.         0.         0.         0.         0.
 1.         0.         0.         0.         0.         0.
 0.         1.         0.         0.         0.         0.
 0.         0.         1.         0.         0.         0.
 0.         0.         1.         0.         0.         0.
 0.         1.         0.         0.         0.         0.
 0.         1.         0.         0.         0.         0.
 1.         0.         0.         0.         0.         0.
 0.         1.         0.         0.         0.

# Now that you have X_onehot, X_kmer, X_binary, X_full, and the target y, let’s add code to export them as files so you can download and use them in ML pipelines.

In [15]:
# Save NumPy arrays
np.save("../data/feature_vectors/X_onehot.npy", X_onehot)
np.save("../data/feature_vectors/X_kmer.npy",   X_kmer)
np.save("../data/feature_vectors/X_binary.npy", X_binary)
np.save("../data/feature_vectors/X_full.npy",   X_full)

print("Saved feature arrays as .npy files")


Saved feature arrays as .npy files


In [17]:
# Save as Pythonic, or tabular format:
# One-hot features
pd.DataFrame(X_onehot).assign(label=y).to_csv("../data/feature_vectors/X_onehot.tsv", sep="\t", index=False)

# k-mer features
pd.DataFrame(X_kmer).assign(label=y).to_csv("../data/feature_vectors/X_kmer.tsv", sep="\t", index=False)

# binary features
pd.DataFrame(X_binary).assign(label=y).to_csv("../data/feature_vectors/X_binary.tsv", sep="\t", index=False)

# full feature matrix
pd.DataFrame(X_full).assign(label=y).to_csv("../data/feature_vectors/X_full.tsv", sep="\t", index=False)

print("Saved feature matrices as TSV files")


Saved feature matrices as TSV files


In [4]:
ALPHABET = "acguxz"   # a,c,g,u + x(unknown) + z(pad)
IDX = {ch:i for i,ch in enumerate(ALPHABET)}
SEQ_LEN = 24
V = len(ALPHABET)

def one_hot_seq(seq: str) -> np.ndarray:
    """Return flattened one-hot: (SEQ_LEN * V,)"""
    arr = np.zeros((SEQ_LEN, V), dtype=np.float32)
    for i, ch in enumerate(seq):
        j = IDX.get(ch)
        if j is not None:
            arr[i, j] = 1.0
    return arr.ravel()

# Encode sense and antisense
X_sense = np.stack([one_hot_seq(s) for s in df["sense"].astype(str).values])      # shape: (N, 24*6)
X_anti  = np.stack([one_hot_seq(s) for s in df["antisense"].astype(str).values])  # shape: (N, 24*6)

# Concatenate features: sense || antisense
X_onehot = np.concatenate([X_sense, X_anti], axis=1)   # shape: (N, 24*6*2) = (N, 288)
y = df["Efficiacy"].astype(float).to_numpy()

print("X_sense:", X_sense.shape, "X_anti:", X_anti.shape)
print("X_onehot:", X_onehot.shape, "y:", y.shape)


X_sense: (3991, 144) X_anti: (3991, 144)
X_onehot: (3991, 288) y: (3991,)


In [ ]:
Add sequence-derived features and then combine them with one-hot encoding

1) k-mer counts (mono/di/tri) per strand (ignoring pad z windows)

2) GC & composition metrics

3) Binary chemophysical patterns (R/Y, S/W, M/K)

4) Concatenate sense || antisense, and (optionally) stack with one-hot to get a single X_full

In [5]:
# Helpers for k-mers, GC, binary patterns

def kmer_vocab(k, alphabet=ALPH_COUNT):
    return ["".join(p) for p in itertools.product(alphabet, repeat=k)]

def kmer_counts(seq: str, k: int, alphabet=ALPH_COUNT, skip_with_z=True) -> np.ndarray:
    """Raw k-mer counts; skip windows containing 'z' if skip_with_z."""
    vocab = kmer_vocab(k, alphabet)
    idx = {kmer:i for i,kmer in enumerate(vocab)}
    counts = np.zeros(len(vocab), dtype=np.float32)
    L = len(seq)
    for i in range(L - k + 1):
        w = seq[i:i+k]
        if skip_with_z and ("z" in w):
            continue
        # ensure only allowed alphabet (map others to x if any slipped through)
        w = re.sub(rf"[^{alphabet}]", "x", w)
        j = idx.get(w)
        if j is not None:
            counts[j] += 1.0
    return counts  # keep raw counts (you can normalize later)

def gc_metrics(seq: str) -> np.ndarray:
    """[gc_count, len_no_z, gc_pct_on_len, gc_pct_on_acgt, x_frac_no_z, z_frac]"""
    s = seq.replace("z", "")
    n_total = len(s)
    n_acgt = sum(ch in "acgu" for ch in s)
    gc = sum(ch in "gc" for ch in s)
    return np.array([
        float(gc),
        float(n_total),
        (gc / n_total) if n_total else 0.0,
        (gc / n_acgt) if n_acgt else 0.0,
        (s.count("x") / n_total) if n_total else 0.0,
        (seq.count("z") / len(seq)) if len(seq) else 0.0,
    ], dtype=np.float32)

def binary_patterns(seq: str) -> np.ndarray:
    """Fractions over non-pad positions: [R,Y,S,W,M,K]"""
    s = seq.replace("z", "")
    L = len(s) or 1
    R = sum(ch in "ag" for ch in s) / L
    Y = sum(ch in "cu" for ch in s) / L
    S = sum(ch in "gc" for ch in s) / L
    W = sum(ch in "au" for ch in s) / L
    M = sum(ch in "ac" for ch in s) / L
    K = sum(ch in "gu" for ch in s) / L
    return np.array([R, Y, S, W, M, K], dtype=np.float32)


NameError: name 'ALPH_COUNT' is not defined